# Pulmonary Nodule Detection from CT (3D)

Volumetric (3D) detection of pulmonary nodules using LUNA16-style data: load 3D patches around candidate locations and train a 3D CNN to classify nodule vs non-nodule.

In [ ]:
# Run this cell once to install dependencies (then restart kernel if needed)
!pip install SimpleITK pandas numpy torch torchvision scikit-learn tqdm matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import SimpleITK as sitk
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Config and paths

In [ ]:
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "dataset_export").exists():
    PROJECT_ROOT = Path("d:/mlmed")  # run notebook from project root, or set this to your path
DATA_DIR = PROJECT_ROOT / "dataset_export"
PATCH_SIZE = 32          # 32^3 voxel patch
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3
# HU window for lung CT (normalize to ~0-1)
HU_MIN, HU_MAX = -1000, 400
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 3D patch dataset (load .mhd, crop around candidate)

In [ ]:
def load_volume_mhd(mhd_path, root: Path):
    """Load a CT volume from .mhd; return array (z,y,x) and origin, spacing (x,y,z).
    SimpleITK finds the .raw file relative to the .mhd's directory, so we switch cwd when reading.
    """
    full_path = (root / mhd_path).resolve()
    if not full_path.exists():
        raise FileNotFoundError(f"MHD not found: {full_path}")
    mhd_dir = full_path.parent
    orig_cwd = os.getcwd()
    try:
        os.chdir(mhd_dir)
        img = sitk.ReadImage(str(full_path))
        arr = np.asarray(sitk.GetArrayFromImage(img), dtype=np.float32)
        origin = np.array(img.GetOrigin())   # (x,y,z)
        spacing = np.array(img.GetSpacing()) # (x,y,z)
        return arr, origin, spacing
    finally:
        os.chdir(orig_cwd)

def world_to_voxel(world_xyz, origin, spacing):
    """Convert world (x,y,z) to voxel indices. Array is (z,y,x) so return (z,y,x)."""
    vox_xyz = (np.asarray(world_xyz) - origin) / spacing
    return (vox_xyz[2], vox_xyz[1], vox_xyz[0])  # z, y, x for numpy

def normalize_hu(patch: np.ndarray, hu_min=HU_MIN, hu_max=HU_MAX):
    """Clip to lung window and normalize to [0,1]."""
    patch = np.clip(patch, hu_min, hu_max)
    return (patch - hu_min) / (hu_max - hu_min)

class NodulePatchDataset(Dataset):
    _volume_cache = {}  # shared cache across instances: seriesuid -> (arr, origin, spacing) or None

    def __init__(self, candidates_df: pd.DataFrame, manifest_df: pd.DataFrame, root: Path, patch_size: int):
        self.root = Path(root)
        self.patch_size = patch_size
        self.manifest = manifest_df.set_index("seriesuid")["mhd_path"].to_dict()
        self.candidates = candidates_df.reset_index(drop=True)
        self.half = patch_size // 2

    def __len__(self):
        return len(self.candidates)

    def _get_volume(self, seriesuid, mhd_path):
        if seriesuid in NodulePatchDataset._volume_cache:
            return NodulePatchDataset._volume_cache[seriesuid]
        try:
            data = load_volume_mhd(mhd_path, self.root)
            NodulePatchDataset._volume_cache[seriesuid] = data
            return data
        except Exception as e:
            NodulePatchDataset._volume_cache[seriesuid] = None
            return None

    def __getitem__(self, idx):
        row = self.candidates.iloc[idx]
        seriesuid = row["seriesuid"]
        world = (row["coordX"], row["coordY"], row["coordZ"])
        label = int(row["class"])
        mhd_path = self.manifest.get(seriesuid)
        if mhd_path is None:
            return torch.zeros(1, self.patch_size, self.patch_size, self.patch_size), label
        vol = self._get_volume(seriesuid, mhd_path)
        if vol is None:
            return torch.zeros(1, self.patch_size, self.patch_size, self.patch_size), label
        arr, origin, spacing = vol
        vz, vy, vx = world_to_voxel(world, origin, spacing)
        z, y, x = int(round(vz)), int(round(vy)), int(round(vx))
        z1, z2 = max(0, z - self.half), min(arr.shape[0], z + self.half)
        y1, y2 = max(0, y - self.half), min(arr.shape[1], y + self.half)
        x1, x2 = max(0, x - self.half), min(arr.shape[2], x + self.half)
        patch = arr[z1:z2, y1:y2, x1:x2]
        # Crop to at most patch_size per dim (handles non-standard volume shapes/axis order)
        patch = patch[: self.patch_size, : self.patch_size, : self.patch_size]
        # Pad to fixed size if needed (boundary or small volume)
        out = np.zeros((self.patch_size, self.patch_size, self.patch_size), dtype=np.float32)
        d0, d1, d2 = min(patch.shape[0], self.patch_size), min(patch.shape[1], self.patch_size), min(patch.shape[2], self.patch_size)
        out[:d0, :d1, :d2] = patch[:d0, :d1, :d2]
        out = normalize_hu(out)
        out = out[np.newaxis, ...]  # (1, D, H, W)
        return torch.from_numpy(out).float(), label

## 3D CNN model (patch classifier)

In [ ]:
class Conv3dBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Conv3d(in_c, out_c, 3, padding=1)
        self.bn = nn.BatchNorm3d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool3d(2)

    def forward(self, x):
        return self.pool(self.relu(self.bn(self.conv(x))))

class NoduleNet3D(nn.Module):
    """Small 3D CNN: 32^3 -> binary nodule vs non-nodule."""
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            Conv3dBlock(in_channels, 32),   # 32^3 -> 16^3
            Conv3dBlock(32, 64),            # 16^3 -> 8^3
            Conv3dBlock(64, 128),           # 8^3 -> 4^3
            nn.AdaptiveAvgPool3d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = NoduleNet3D().to(DEVICE)
print(model)

## Load data and create dataloaders (balanced sampling)

In [ ]:
root = PROJECT_ROOT
manifest = pd.read_csv(DATA_DIR / "series_manifest.csv")
train_df = pd.read_csv(DATA_DIR / "candidates_train.csv")
val_df = pd.read_csv(DATA_DIR / "candidates_val.csv")
test_df = pd.read_csv(DATA_DIR / "candidates_test.csv")

# Use balanced subset for faster training (optional: use candidates_train.csv for full data)
train_balanced_path = DATA_DIR / "candidates_train_balanced_subset.csv"
if train_balanced_path.exists():
    train_df = pd.read_csv(train_balanced_path)
    print("Using balanced train subset:", len(train_df))
else:
    print("Using full train:", len(train_df))

train_ds = NodulePatchDataset(train_df, manifest, root, PATCH_SIZE)
val_ds = NodulePatchDataset(val_df, manifest, root, PATCH_SIZE)
test_ds = NodulePatchDataset(test_df, manifest, root, PATCH_SIZE)

# Weighted sampling to balance classes when using full (imbalanced) train
if len(train_df) > 5000:  # full train
    labels = train_ds.candidates["class"].values
    class_counts = np.bincount(labels)
    weights = 1.0 / class_counts[labels]
    sampler = WeightedRandomSampler(weights, len(weights))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Training loop

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses, val_losses, val_aucs = [], [], []

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for patches, labels_b in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        patches, labels_b = patches.to(DEVICE), labels_b.to(DEVICE).float().unsqueeze(1)
        optimizer.zero_grad()
        logits = model(patches)
        loss = criterion(logits, labels_b)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_losses.append(running_loss / len(train_loader))
    scheduler.step()

    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for patches, labels_b in val_loader:
            patches, labels_b = patches.to(DEVICE), labels_b.to(DEVICE).float().unsqueeze(1)
            logits = model(patches)
            loss = criterion(logits, labels_b)
            val_loss += loss.item()
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            all_preds.extend(probs)
            all_labels.extend(labels_b.cpu().numpy().ravel())
    val_losses.append(val_loss / len(val_loader))
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    if all_labels.sum() > 0 and (1 - all_labels).sum() > 0:
        auc = roc_auc_score(all_labels, all_preds)
    else:
        auc = 0.0
    val_aucs.append(auc)
    print(f"Epoch {epoch+1}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}  val_AUC={auc:.4f}")

Epoch 1/10:   7%|▋         | 3/43 [00:15<03:21,  5.03s/it]

## Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(train_losses, label="Train loss")
ax1.plot(val_losses, label="Val loss")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.set_title("Loss")
ax2.plot(val_aucs, color="green")
ax2.set_xlabel("Epoch")
ax2.set_title("Validation AUC")
plt.tight_layout()
plt.show()

## Test set evaluation

In [ ]:
# Free RAM before test (CT volumes are cached; full test can glitch on limited memory)
NodulePatchDataset._volume_cache.clear()
model.eval()
all_preds, all_labels = [], []
total_batches = TEST_MAX_BATCHES if TEST_MAX_BATCHES is not None else len(test_loader)
with torch.no_grad():
    for batch_idx, (patches, labels_b) in enumerate(tqdm(test_loader, desc="Test", total=total_batches)):
        if TEST_MAX_BATCHES is not None and batch_idx >= TEST_MAX_BATCHES:
            break
        patches = patches.to(DEVICE)
        logits = model(patches)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        all_preds.extend(probs)
        all_labels.extend(labels_b.numpy())
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
pred_binary = (all_preds >= 0.5).astype(int)
if TEST_MAX_BATCHES is not None:
    print(f"(Limited to {len(all_labels)} samples. Set TEST_MAX_BATCHES = None in config for full test.)")
print("Confusion matrix:")
print(confusion_matrix(all_labels, pred_binary))
print("\nClassification report:")
print(classification_report(all_labels, pred_binary, target_names=["Non-nodule", "Nodule"], zero_division=0))
if all_labels.sum() > 0 and (1 - all_labels).sum() > 0:
    print(f"Test ROC-AUC: {roc_auc_score(all_labels, all_preds):.4f}")

## Save model (optional)

In [ ]:
# torch.save(model.state_dict(), PROJECT_ROOT / "nodule_3dcnn.pt")